In [ ]:
#Replace with your directory

source("C:/Users/grifclay/Desktop/Projects/Spatial_Transcriptomics_Analysis/SeuratCropper.R")
library(data.table)
library(sf)
library(geojsonsf)
library(Seurat)
library(arrow)
library(dplyr)
library(cowplot)
library(ggplot2)

In [ ]:
home = "C:/Users/grifclay/Desktop/Projects/Spatial_Transcriptomics_Analysis/data/Male"
save.location = "C:/Users/grifclay/Desktop/Projects/Spatial_Transcriptomics_Analysis/Routput"
setwd(home)
#Replace with parent directory of Xenium data 
filename = "20251113_MS_Sag16_ID57700"

obj <- LoadXenium(filename, fov="fov")
#Filter out cells with 0 count
obj <- subset(obj, subset = nCount_Xenium > 0)

In [ ]:
saveRDS(obj, file="Male_Sag_16.rds")

In [ ]:
obj <- readRDS("C:/Users/grifclay/Desktop/Projects/Spatial_Transcriptomics_Analysis/Routput/Male_Sag_16")

In [ ]:


#Location of .geoJSON file with cropping polygon
polygon_file = "C:/Users/grifclay/Desktop/Projects/Spatial_Transcriptomics_Analysis/data/cropping/coords/male_sag16_ATN_formatted.geojson"

Polygon_ATN <- geojson_sf(polygon_file)


In [ ]:
#Created cropped Seurat object
ATN_obj = PolygonCropSeurat(obj, list(fov = Polygon_ATN))

In [ ]:
ATN_obj <- SCTransform(ATN_obj, assay = "Xenium")

In [ ]:
ATN_obj = RunPCA(ATN_obj, npcs = 30)
ElbowPlot(ATN_obj, ndims = 30, reduction = "pca")

In [ ]:
ATN_obj <- RunUMAP(ATN_obj, dims = 1:12)
ATN_obj <- FindNeighbors(ATN_obj, dims = 1:12)

saveRDS(ATN_obj, file = "Male_Sag_16_ATN.rds")

In [ ]:
ATN_obj <- FindClusters(ATN_obj, resolution = 0.07)


In [ ]:
DimPlot(ATN_obj, label = TRUE)

In [ ]:
p = ImageDimPlot(ATN_obj, fov = "fov")
ggsave(filename = "Male_Sag_16_ATN_plot.tiff",
       plot = p,
       width = 12, height = 10, units = "in", dpi = 600)

In [ ]:
p

In [ ]:
#Identifying most differentiated genes between clusters
#Tested roc and wilcox tests, wilcox gave better differentiated genes for cluster 1
markers <- FindAllMarkers(ATN_obj, 
                            test.use = "wilcox",
                            only.pos = TRUE, 
                            min.pct = 0.25, 
                            logfc.threshold = 0.25)

In [ ]:
#Confirm column names of markers
colnames(markers)

In [ ]:
#Filter for high-confidence markers:

filtered_markers <- markers %>%
  filter(
    p_val_adj  < .05,        # Keep only statistically significant genes
    avg_log2FC > 0.5,        # Keep genes with a minimum log2FC (adjust as needed)
    pct.1 > 0.3              # Keep genes expressed in at least 30% of the cluster cells (pct.1)
  )

In [ ]:
write.csv(filtered_markers, "top_cluster_markers_wilcox.csv", row.names = FALSE)

In [ ]:
top_genes_to_plot <- filtered_markers %>%
  group_by(cluster) %>%
  arrange(desc(avg_log2FC)) %>%
  slice_head(n = 2) %>%
  # Extract only the gene names into a simple vector
  pull(gene)

# Check the resulting vector of gene names
print(top_genes_to_plot)

In [ ]:
vp <-VlnPlot(
  object = ATN_obj,
  features = top_genes_to_plot,
  # Plot each cluster on the x-axis, using its identity (the default cluster assignment)
)

ggsave(filename = "ViolinPlotTopGenesATN_roc.png", plot = vp)

In [ ]:
vp

In [ ]:
#Cluster 0
# 1. Identify cells that express Hcn1 OR Grm3 (expression > 0 in either) 
relevant_cells <- WhichCells(ATN_obj, expression = Hcn1 > 0 | Scn4b > 0)

# 1. Run the function with combine = FALSE to get a list of plots
plot_list <- ImageFeaturePlot(
  ATN_obj, 
  fov = "fov", 
  features = c("Hcn1", "Grm3"), 
  blend = TRUE,
  blend.threshold = .5,
  combine = FALSE,
  cells = relevant_cells
)

# 2. Print only the 3rd plot (the blended overlay)
cluster_0_plot <- plot_list[[3]] + ggtitle("Cluster 0: Hcn1 & Grm3")
ggsave(filename = "Cluster0_FeaturePlot.png", 
  plot= cluster_0_plot)

#------------------------

#cluster 1
relevant_cells <- WhichCells(ATN_obj, expression = Agt > 0 | S1pr1 > 0)

# 1. Run the function with combine = FALSE to get a list of plots
plot_list <- ImageFeaturePlot(
  ATN_obj, 
  fov = "fov", 
  features = c("Agt", "S1pr1"), 
  blend = TRUE,
  blend.threshold = .5,
  combine = FALSE,
  cells = relevant_cells
)

# 2. Print only the 3rd plot (the blended overlay)
cluster_1_plot <- plot_list[[3]]+ ggtitle("Cluster 1: Agt & S1pr1")
ggsave(filename = "Cluster1_FeaturePlot.png",
  plot = cluster_1_plot)

#---------------------

#cluster 2
relevant_cells <- WhichCells(ATN_obj, expression = Mag > 0 | Mog > 0)

plot_list <- ImageFeaturePlot(
  ATN_obj, 
  fov = "fov", 
  features = c("Mag", "Mog"), 
  blend = TRUE,
  blend.threshold = .5,
  combine = FALSE,
  cells = relevant_cells
)

cluster_2_plot <- plot_list[[3]] + ggtitle("Cluster 2: Mag & Mog")
ggsave(filename = "Cluster_2_FeaturePlot.png",
  plot = cluster_2_plot)

#------------------

#cluster 3
relevant_cells <- WhichCells(ATN_obj, expression = Cbln1 > 0 | Kcnf1 > 0)

plot_list <- ImageFeaturePlot(
  ATN_obj, 
  fov = "fov", 
  features = c("Cbln1", "Kcnf1"), 
  blend = TRUE,
  blend.threshold = .5,
  combine = FALSE,
  cells = relevant_cells
)

cluster_3_plot <- plot_list[[3]] + ggtitle("Cluster 3: Cbln1 & Kcnf1")
ggsave(filename="Cluster_3_FeaturePlot.png",
  plot = cluster_3_plot)

#------------

#Cluster 4

relevant_cells <- WhichCells(ATN_obj, expression = Cfap65 > 0 | Stoml3 > 0)

plot_list <- ImageFeaturePlot(
  ATN_obj, 
  fov = "fov", 
  features = c("Cfap65", "Stoml3"), 
  blend = TRUE,
  blend.threshold = .5,
  combine = FALSE,
  cells = relevant_cells
)

cluster_4_plot <- plot_list[[3]] + ggtitle("Cluster 4: Cfap65 & Stoml3")
ggsave(filename="Cluster_4_FeaturePlot.png",
  plot = cluster_4_plot)

